In [1]:
 #Verify GPU
 import os
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

NVIDIA A100-SXM4-40GB, 40960 MiB


In [ ]:
# ============================================================
# GCS Authentication (Permanent — Service Account)
# WITH NO COMMENTS FOR EACH LINE
# ============================================================
import subprocess, threading, time, os

KEY_FILE   = "/content/gcs-key.json"
PROJECT_ID = "solar-cycle-487619-t6"
GCS_BUCKET = "gs://mywaymo-perdataset-2026"

def _reauth():
    subprocess.run(["gcloud", "auth", "activate-service-account",
                    "--key-file", KEY_FILE], capture_output=True)
    subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID],
                    capture_output=True)

def start_auth_keepalive(interval_minutes=45):
    def _loop():
        while True:
            time.sleep(interval_minutes * 60)
            _reauth()
            print(f"🔄 GCS re-auth at {time.strftime('%H:%M:%S')}")
    threading.Thread(target=_loop, daemon=True).start()
    print(f"🔄 Keepalive started (every {interval_minutes} min)")

# Step 1 — bootstrap: interactive login just to grab the key
!gcloud auth login --no-launch-browser
!gsutil cp gs://mywaymo-perdataset-2026/auth/gcs-key.json /content/gcs-key.json

# Step 2 — switch to service account (never expires)
_reauth()
start_auth_keepalive()

# Step 3 — verify
result = subprocess.run(["gsutil", "ls", GCS_BUCKET], capture_output=True, text=True)
print("✅ GCS authenticated and connected" if result.returncode == 0 else f"❌ {result.stderr}")

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=nhQGKAsug7NINDgbU9a3ckmvIR5NKT&prompt=consent&token_usage=remote&access_type=offline&code_challenge=VSoVLoZRyjlwFx8nSfQtxn0LyKVJQ7ovKpLoTgUMdZI&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0Aci98E8hIFhMaLZfFKT4UDLDBUKzsj3TBiqIOiV8efA7UoLQEb6leloZFtR-r5jJIuB0mw

You are now logged in as [suh2162674@maricopa.edu].
Your current projec

In [2]:
# ============================================================
# GCS Authentication (Permanent — Service Account)
# ============================================================

#subprocess — Python run shell commands (like gcloud, gsutil) from inside Python
#threading — Python run a background task simultaneously while training runs
#time — used for time.sleep() (pause) and time.strftime() (print current time)
#os — used for file checks like os.path.exists()

import subprocess, threading, time, os

#below 3 variable definitions — store our paths/IDs
KEY_FILE   = "/content/gcs-key.json"
PROJECT_ID = "solar-cycle-487619-t6"
GCS_BUCKET = "gs://mywaymo-perdataset-2026"

#Defines a reusable function that re-authenticates GCS
def _reauth():
    # tells gcloud "use this service account key for all future GCS operations"
    subprocess.run(["gcloud", "auth", "activate-service-account",
                    "--key-file", KEY_FILE], capture_output=True)
    #— tells gcloud which GCP project to bill/access.
    #capture_output=True — suppresses output so it runs silently in the background without cluttering our notebook
    subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID],
                    capture_output=True)
# calls _reauths() to reauthenticate every 45 minutes by default (tokens expire after ~60 min, so 45 is safe).
#waits 45 minutes (45 × 60 = 2700 seconds)
def start_auth_keepalive(interval_minutes=45):
    #runs forever in a loop
    def _loop():
        while True:
            time.sleep(interval_minutes * 60)
            #silently refreshes the GCS token
            _reauth()
            #shows a timestamp so we know re-auth happened (useful for debugging)
            print(f"🔄 GCS re-auth at {time.strftime('%H:%M:%S')}")
    #threading.Thread(target=_loop) — creates a background thread that runs _loop
    #daemon=True->if the main (colab) program dies, kill this helper (_reauth() thread) too automatically."
    #.start() — start background thread immediately
    threading.Thread(target=_loop, daemon=True).start()
    #print confirms it started
    print(f"🔄 Keepalive started (every {interval_minutes} min)")

# Step 1 — bootstrap: needed interactive login just to grab the key once per session, then we immediately switch to the service account
!gcloud auth login --no-launch-browser #interactive one-time login — opens a URL,paste a code — this auth needed to download the key
#downloads gcs-key.json from my GCS bucket to /content/ on the Colab VM
!gsutil cp gs://mywaymo-perdataset-2026/auth/gcs-key.json /content/gcs-key.json

# Step 2 — immediately switches from the short-lived interactive login to service account (never expires)
_reauth()
start_auth_keepalive()#starts the background thread that re-auths every 45 min for the rest of the session

# Step 3 — verify

#Runs gsutil ls gs://mywaymo-perdataset-2026 to test the connection
#capture_output=True — captures the output instead of printing it directly
#text=True — returns output as a string (not bytes)
#result.returncode == 0 — 0 means success in Linux/shell, anything else is an error
#Prints ✅ if it worked, or the actual error message if it didn't

result = subprocess.run(["gsutil", "ls", GCS_BUCKET], capture_output=True, text=True)
print("✅ GCS authenticated and connected" if result.returncode == 0 else f"❌ {result.stderr}")

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=Zr9BvCZNY1nfYPYIsjxd2t72c0l5hj&prompt=consent&token_usage=remote&access_type=offline&code_challenge=N1vAdmlKhpzCDvqejaMkH7ll2bJ4RuRJECC8gZu3JiE&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0Aci98E9wfkBqIe394cEF1FMfNG12PHqjZzJIylE7TvjsBBnH-IRuagWMZncdZmKG4ZgvQA

You are now logged in as [suh2162674@maricopa.edu].
Your current projec

In [ ]:
## old discard
# ============================================================
# GCS Authentication (Permanent — Service Account)
# Key stored in GCS — downloaded every session — never expires
# ============================================================

#authenticate GCS(by copying link in seperate browser)
!gcloud auth login --no-launch-browser

# Download key from GCS to colab
!gsutil cp gs://mywaymo-perdataset-2026/auth/gcs-key.json /content/gcs-key.json

# Authenticate with service account
!gcloud auth activate-service-account --key-file=/content/gcs-key.json
!gcloud config set project solar-cycle-487619-t6

# Verify
!gsutil ls gs://mywaymo-perdataset-2026/
print("GCS authenticated with service account ✅ (never expires)")

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=05JIiLmbymmLa7CWtWYbNl34g8ZAQs&prompt=consent&token_usage=remote&access_type=offline&code_challenge=uXdZYo5bmejgfVS53hzdOOGVnJgBzzy0Nr2bGsWNNFo&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0Aci98E8_wCoWGIr_QH_aLKn_t4YGHC11VDATi7jzieDBBX5v8M1-s2BaFpjWPr1r5mdLLA

You are now logged in as [suh2162674@maricopa.edu].
Your current projec

In [ ]:
# check the rfdetr version
!pip show rfdetr

Name: rfdetr
Version: 1.4.3
Summary: RF-DETR
Home-page: https://github.com/roboflow/rf-detr
Author: 
Author-email: "Roboflow, Inc" <develop@roboflow.com>
License: Apache License 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: matplotlib, peft, pycocotools, pydantic, requests, rf100vl, roboflow, scipy, supervision, torch, torchvision, tqdm, transformers
Required-by: 


In [3]:
# Install RF-DETR
#must run every session;Colab reinstalls the latest version on each new session.
#This is a non-critical warning:not using torchaudio anywhere in our training pipeline — it's irrelevant
!pip install rfdetr==1.4.3 -q
print("RF-DETR installed ✅; restart session")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 151.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
#find the exact version that saved the checkpoint, check:
!pip index versions rfdetr

rfdetr (1.6.4)
Available versions: 1.6.4, 1.6.3, 1.6.2, 1.6.1, 1.6.0, 1.5.2, 1.5.1, 1.5.0, 1.4.3, 1.4.2, 1.4.1, 1.4.0.post0, 1.3.0, 1.2.1, 1.2.0, 1.1.0, 1.0.8, 1.0.7, 1.0.6, 1.0.5, 1.0.4, 1.0.3, 1.0.2, 1.0.1, 1.0.0, 0.0.1
  INSTALLED: 1.4.3
  LATEST:    1.6.4


In [ ]:
#After restart, verify torch version is correct:
import torch
print(torch.__version__)        # should be 2.8.0
print(torch.cuda.is_available()) # should be True
print(torch.cuda.get_device_name(0)) # should show A100

2.8.0+cu128
True
NVIDIA A100-SXM4-40GB


In [2]:
#Imports
import os
import subprocess
import time
from pathlib import Path
from rfdetr import RFDETRLarge

import csv


print("All imports successful ✅")

All imports successful ✅


In [4]:
# ============================================================
# GLOBAL CONSTANTS — Run this cell first, defines everything
# ============================================================
from pathlib import Path

# Dataset
LOCAL_DATA_DIR      = Path("/content/waymo_yolo")
LOCAL_DATA_DIR.mkdir(exist_ok=True)
CLASS_NAMES         = ["Vehicle", "Pedestrian", "Sign", "Cyclist"]

# Training
LOCAL_OUTPUT_DIR    = "/content/runs/rfdetr_150"
TOTAL_EPOCHS        = 50

# GCS
GCS_BUCKET          = "gs://mywaymo-perdataset-2026"
GCS_CHECKPOINT_PATH = f"{GCS_BUCKET}/checkpoints"
GCS_CSV             = f"{GCS_BUCKET}/models/rfdetr_150_training_history.csv"
GCS_AUTH_KEY        = f"{GCS_BUCKET}/auth/gcs-key.json"

# Local paths
KEY_FILE            = "/content/gcs-key.json"
CSV_PATH            = "/content/training_history.csv"

print("Global constants defined ✅")

Global constants defined ✅


In [5]:
#Restore dataset from GCS

print("Restoring...")
!gsutil -m -q cp -r gs://mywaymo-perdataset-2026/prepared_150/images /content/waymo_yolo/
!gsutil -m -q cp -r gs://mywaymo-perdataset-2026/prepared_150/labels /content/waymo_yolo/
print("Done ✅")

print(f"Train: {len(os.listdir('/content/waymo_yolo/images/train'))}")
print(f"Val: {len(os.listdir('/content/waymo_yolo/images/val'))}")

Restoring...
Done ✅
Train: 23033
Val: 5759


In [ ]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   54G   60G  48% /


In [ ]:
# Check system memory
!free -h

               total        used        free      shared  buff/cache   available
Mem:            83Gi       3.6Gi        44Gi       3.0Mi        35Gi        79Gi
Swap:             0B          0B          0B


In [ ]:
#counts how many images have been downloaded so far.
!ls -la /content/waymo_yolo/images/train/ | wc -l

23036


In [ ]:
#to check labels too:
!ls /content/waymo_yolo/labels/train/ | wc -l

23033


In [6]:
#create data.yaml
yaml_content = """path: /content/waymo_yolo
train: images/train
val: images/val

nc: 4
names:
  - Vehicle
  - Pedestrian
  - Sign
  - Cyclist
"""

yaml_path = LOCAL_DATA_DIR / "data.yaml"
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print("YAML created ✅")
print(open(yaml_path).read())

YAML created ✅
path: /content/waymo_yolo
train: images/train
val: images/val

nc: 4
names:
  - Vehicle
  - Pedestrian
  - Sign
  - Cyclist



In [ ]:
#Check if data.yaml was actually created:

print(os.listdir("/content/waymo_yolo"))
print(open("/content/waymo_yolo/data.yaml").read())

['images', 'labels', 'data.yaml']
path: /content/waymo_yolo
train: images/train
val: images/val

nc: 4
names:
  - Vehicle
  - Pedestrian
  - Sign
  - Cyclist



**Fix Directory Structure with Symlinks**

RF-DETR expects:
```
waymo_yolo/
├── data.yaml
├── train/
│   ├── images/
│   └── labels/
└── valid/
    ├── images/
    └── labels/
```


But GCS data has:
```
waymo_yolo/
├── data.yaml
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/

```

Our GCS dataset uses YOLO convention (images/ and labels/ at top level).
We use symlinks to avoid copying 29k images and it just points valid to our existing val folder.

In [7]:
# Symlinks for images/valid (RF-DETR wants 'valid' not 'val')
if not os.path.exists("/content/waymo_yolo/images/valid"):
    os.symlink("/content/waymo_yolo/images/val", "/content/waymo_yolo/images/valid")
if not os.path.exists("/content/waymo_yolo/labels/valid"):
    os.symlink("/content/waymo_yolo/labels/val", "/content/waymo_yolo/labels/valid")

# Symlinks for top-level train/ and valid/ structure
os.makedirs("/content/waymo_yolo/train", exist_ok=True)
os.makedirs("/content/waymo_yolo/valid", exist_ok=True)

if not os.path.exists("/content/waymo_yolo/train/images"):
    os.symlink("/content/waymo_yolo/images/train", "/content/waymo_yolo/train/images")
if not os.path.exists("/content/waymo_yolo/train/labels"):
    os.symlink("/content/waymo_yolo/labels/train", "/content/waymo_yolo/train/labels")
if not os.path.exists("/content/waymo_yolo/valid/images"):
    os.symlink("/content/waymo_yolo/images/val", "/content/waymo_yolo/valid/images")
if not os.path.exists("/content/waymo_yolo/valid/labels"):
    os.symlink("/content/waymo_yolo/labels/val", "/content/waymo_yolo/valid/labels")

print("Directory structure:")
print(os.listdir("/content/waymo_yolo"))
print(os.listdir("/content/waymo_yolo/train"))
print(os.listdir("/content/waymo_yolo/valid"))
print("Symlinks created ✅")

Directory structure:
['images', 'valid', 'train', 'data.yaml', 'labels']
['images', 'labels']
['images', 'labels']
Symlinks created ✅


In [ ]:
#Check if the full directory structure is now correct:

print(os.listdir("/content/waymo_yolo"))
print(os.listdir("/content/waymo_yolo/images"))
print(os.listdir("/content/waymo_yolo/labels"))

['train', 'images', 'labels', 'data.yaml', 'valid']
['train', 'val', 'valid']
['train', 'val', 'valid']


In [8]:
# ============================================================
# Check for Existing GCS Checkpoints (Resume Support)
# If a previous run was interrupted, find the last saved epoch
# ============================================================
result = subprocess.run(
    ["gsutil", "ls", "gs://mywaymo-perdataset-2026/checkpoints/"],
    capture_output=True, text=True
)

checkpoints = [line for line in result.stdout.strip().split('\n') if 'rfdetr_epoch_' in line]
checkpoints.sort(key=lambda x: int(x.split('rfdetr_epoch_')[1].replace('.pth', '')))

if checkpoints:
    print(f"Found {len(checkpoints)} existing checkpoints:")
    for cp in checkpoints:
        print(f"  {cp}")
    last_checkpoint = checkpoints[-1]
    last_epoch = int(last_checkpoint.split('rfdetr_epoch_')[1].replace('.pth', ''))
    print(f"\nLast saved epoch: {last_epoch}")
    print(f"Can resume from: {last_checkpoint}")
else:
    print("No existing checkpoints found — starting fresh from epoch 1")
    last_epoch = 0

Found 53 existing checkpoints:
  gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_0.pth
  gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_1.pth
  gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_2.pth
  gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_3.pth
  gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_4.pth
  gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_5.pth
  gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_6.pth
  gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_7.pth
  gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_8.pth
  gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_9.pth
  gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_10.pth
  gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_11.pth
  gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_12.pth
  gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_13.pth
  gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_14.pth
  gs://mywaymo-perdataset-2026/che

In [ ]:
# ============================================================
# Resume from Last Checkpoint (if available)
# Skip this cell if starting fresh
# ============================================================
RESUME_CHECKPOINT = None

if last_epoch > 0:
    local_checkpoint = f"/content/rfdetr_epoch_{last_epoch}.pth"
    print(f"Downloading checkpoint from epoch {last_epoch}...")
    subprocess.run([
        "gsutil", "cp",
        f"gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_{last_epoch}.pth",
        local_checkpoint
    ])
    RESUME_CHECKPOINT = local_checkpoint
    print(f"Resume checkpoint ready: {RESUME_CHECKPOINT} ✅")
else:
    print("Starting fresh — no resume needed ✅")

Resume checkpoint ready: /content/rfdetr_epoch_48.pth ✅


def save_to_gcs(metrics):

    epoch = metrics.get("epoch")  

'metrics' is just a dictionary name that RF-DETR passes


RF-DETR actually passes

{
    "epoch": 1,

    "loss": 7.56,

    "mAP": 0.251,

    ...
}

RF-DETR builds that dictionary inside train_from_config and passes it when calling each on_fit_epoch_end callback.

This is the complete final cell with:

✅ All imports

✅ All constants

✅ CSV append per epoch

✅ GCS checkpoint save per epoch

✅ Service account re-auth

✅ Resume support

✅ Optimized training params

✅ model.train() at the end

In [ ]:
#run RF-DETR training:
# ============================================================
# TRAINING (CRASH-SAFE WITH GCS CHECKPOINTING)
#
# Key fixes vs previous run:
# 1. GCS save via on_fit_epoch_end callback (correct RF-DETR hook)
# 2. Callback receives a 'metrics' dict (NOT a trainer object)
#    Correct: def save_to_gcs(metrics):
#    Wrong:   def save_to_gcs(trainer):  <-- silently failed before
# 3. Appended to model.callbacks['on_fit_epoch_end'] BEFORE train()
# 4. try/except ensures GCS failure won't crash training
# 5. timeout=60 prevents GCS hanging
# ============================================================

# ---- Crash-safe GCS callback ----
def save_to_gcs(metrics):
    try:
        # Re-auth with service account
        subprocess.run(
            ["gcloud", "auth", "activate-service-account", "--key-file", KEY_FILE],
            capture_output=True, timeout=30
        )

        epoch = metrics.get("epoch", 0) + last_epoch + 1

        # ---- 1. Append metrics row directly to CSV ----
        file_exists = os.path.exists(CSV_PATH)
        with open(CSV_PATH, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=metrics.keys())
            if not file_exists:
                writer.writeheader()
            writer.writerow(metrics)
        print(f"✅ Epoch {epoch} metrics appended to CSV")

        # ---- 2. Save CSV to GCS ----
        subprocess.run(
            ["gsutil", "-q", "cp", CSV_PATH, GCS_CSV],
            capture_output=True, timeout=30
        )
        print(f"✅ Epoch {epoch} CSV saved to GCS")

        # ---- 3. Save checkpoint to GCS ----
        local_ckpt = f"{LOCAL_OUTPUT_DIR}/checkpoint.pth"
        if not os.path.exists(local_ckpt):
            print(f"⚠️ No checkpoint file found, skipping")
            return
        result = subprocess.run(
            ["gsutil", "-q", "cp", local_ckpt,
             f"{GCS_CHECKPOINT_PATH}/rfdetr_epoch_{epoch}.pth"],
            capture_output=True, timeout=60
        )
        if result.returncode == 0:
            print(f"✅ Epoch {epoch} checkpoint saved to GCS")
        else:
            print(f"⚠️ Checkpoint save failed: {result.stderr.decode()}")

    except Exception as e:
        print(f"⚠️ GCS exception (training continues): {e}")

# ---- Load model ----
#added this environment variable to help with memory fragmentation:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if RESUME_CHECKPOINT:
    print(f"Loading from checkpoint: {RESUME_CHECKPOINT}")
    os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
    model = RFDETRLarge(pretrain_weights=RESUME_CHECKPOINT)
else:
    print("Loading pretrained RF-DETR-L weights...")
    model = RFDETRLarge()

# ---- Attach callback ----
model.callbacks["on_fit_epoch_end"].append(save_to_gcs)
print("GCS callback attached ✅")
print(f"Callbacks registered: {[f.__name__ for f in model.callbacks['on_fit_epoch_end']]}")

# ---- Start training ----
model.train(
    dataset_dir="/content/waymo_yolo",
    dataset_file="yolo",
    epochs=TOTAL_EPOCHS,
    batch_size=2,
    grad_accum_steps=8,
    lr=1e-4,
    lr_encoder=1.5e-4,
    output_dir=LOCAL_OUTPUT_DIR,
    checkpoint_interval=1,
    num_workers=0,
    fp16_eval=False,      # faster evaluation
    early_stopping=True,
    early_stopping_patience=10,
    use_ema=True,
    tensorboard=False,
)

print("Training complete ✅")

[2026-04-14 16:07:15] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-04-14 16:07:15] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


Loading from checkpoint: /content/rfdetr_epoch_48.pth
[2026-04-14 16:07:15] [INFO] rf-detr - Loading pretrain weights


[2026-04-14 16:07:16] [WARNING] rf-detr - Reinitializing detection head with 3 classes based on pretrained weights, configured for 90.
[2026-04-14 16:07:16] [WARNING] rf-detr - Reinitializing your detection head with 4 classes.


GCS callback attached ✅
Callbacks registered: ['save_to_gcs']
[2026-04-14 16:07:16] [INFO] rf-detr - Not using distributed mode
[2026-04-14 16:07:16] [INFO] rf-detr - git:
  unknown

[2026-04-14 16:07:16] [INFO] rf-detr - Namespace(num_classes=4, grad_accum_steps=8, print_freq=10, amp=True, lr=0.0001, lr_encoder=0.00015, batch_size=2, weight_decay=0.0001, epochs=50, lr_drop=100, clip_max_norm=0.1, lr_vit_layer_decay=0.8, lr_component_decay=0.7, do_benchmark=False, dropout=0, drop_path=0.0, drop_mode='standard', drop_schedule='constant', cutoff_epoch=0, pretrained_encoder=None, pretrain_weights='/content/rfdetr_epoch_48.pth', pretrain_exclude_keys=None, pretrain_keys_modify_to_load=None, pretrained_distiller=None, encoder='dinov2_windowed_small', vit_encoder_num_layers=12, window_block_indexes=None, position_embedding='sine', out_feature_indexes=[3, 6, 9, 12], freeze_encoder=False, layer_norm=True, rms_norm=False, backbone_lora=False, force_no_pretrain=False, dec_layers=4, dim_feedforwa

[2026-04-14 16:15:38] [INFO] rf-detr - Epoch: [0]  [   0/1439]  eta: 2:11:43  lr: 0.000100  class_error: 0.00  loss: 4.4785 (4.4785)  loss_ce: 0.4584 (0.4584)  loss_bbox: 0.0301 (0.0301)  loss_giou: 0.3478 (0.3478)  loss_ce_0: 0.5211 (0.5211)  loss_bbox_0: 0.0367 (0.0367)  loss_giou_0: 0.3859 (0.3859)  loss_ce_1: 0.4830 (0.4830)  loss_bbox_1: 0.0350 (0.0350)  loss_giou_1: 0.3687 (0.3687)  loss_ce_2: 0.4605 (0.4605)  loss_bbox_2: 0.0318 (0.0318)  loss_giou_2: 0.3502 (0.3502)  loss_ce_enc: 0.5151 (0.5151)  loss_bbox_enc: 0.0404 (0.0404)  loss_giou_enc: 0.4139 (0.4139)  loss_ce_unscaled: 0.4584 (0.4584)  class_error_unscaled: 0.0000 (0.0000)  loss_bbox_unscaled: 0.0060 (0.0060)  loss_giou_unscaled: 0.1739 (0.1739)  cardinality_error_unscaled: 3877.0000 (3877.0000)  loss_ce_0_unscaled: 0.5211 (0.5211)  loss_bbox_0_unscaled: 0.0073 (0.0073)  loss_giou_0_unscaled: 0.1929 (0.1929)  cardinality_error_0_unscaled: 3877.0000 (3877.0000)  loss_ce_1_unscaled: 0.4830 (0.4830)  loss_bbox_1_unscaled: 

...

[2026-04-14 19:48:23] [INFO] rf-detr - Epoch: [2]  [   0/1439]  eta: 1:15:54  lr: 0.000100  class_error: -0.00  loss: 5.5182 (5.5182)  loss_ce: 0.5357 (0.5357)  loss_bbox: 0.0465 (0.0465)  loss_giou: 0.4870 (0.4870)  loss_ce_0: 0.5557 (0.5557)  loss_bbox_0: 0.0533 (0.0533)  loss_giou_0: 0.5248 (0.5248)  loss_ce_1: 0.5410 (0.5410)  loss_bbox_1: 0.0481 (0.0481)  loss_giou_1: 0.5076 (0.5076)  loss_ce_2: 0.5340 (0.5340)  loss_bbox_2: 0.0476 (0.0476)  loss_giou_2: 0.4923 (0.4923)  loss_ce_enc: 0.5662 (0.5662)  loss_bbox_enc: 0.0536 (0.0536)  loss_giou_enc: 0.5248 (0.5248)  loss_ce_unscaled: 0.5357 (0.5357)  class_error_unscaled: -0.0000 (-0.0000)  loss_bbox_unscaled: 0.0093 (0.0093)  loss_giou_unscaled: 0.2435 (0.2435)  cardinality_error_unscaled: 3860.5000 (3860.5000)  loss_ce_0_unscaled: 0.5557 (0.5557)  loss_bbox_0_unscaled: 0.0107 (0.0107)  loss_giou_0_unscaled: 0.2624 (0.2624)  cardinality_error_0_unscaled: 3872.0000 (3872.0000)  loss_ce_1_unscaled: 0.5410 (0.5410)  loss_bbox_1_unscale

[2026-04-14 21:20:43] [INFO] rf-detr - Test:  [   0/2880]  eta: 0:19:47  class_error: 0.00  loss: 6.2458 (6.2458)  loss_ce: 0.5859 (0.5859)  loss_bbox: 0.0273 (0.0273)  loss_giou: 0.5849 (0.5849)  loss_ce_0: 0.6445 (0.6445)  loss_bbox_0: 0.0289 (0.0289)  loss_giou_0: 0.6071 (0.6071)  loss_ce_1: 0.6094 (0.6094)  loss_bbox_1: 0.0281 (0.0281)  loss_giou_1: 0.5873 (0.5873)  loss_ce_2: 0.6055 (0.6055)  loss_bbox_2: 0.0277 (0.0277)  loss_giou_2: 0.5783 (0.5783)  loss_ce_enc: 0.6250 (0.6250)  loss_bbox_enc: 0.0339 (0.0339)  loss_giou_enc: 0.6720 (0.6720)  loss_ce_unscaled: 0.5859 (0.5859)  class_error_unscaled: 0.0000 (0.0000)  loss_bbox_unscaled: 0.0055 (0.0055)  loss_giou_unscaled: 0.2924 (0.2924)  cardinality_error_unscaled: 265.5000 (265.5000)  loss_ce_0_unscaled: 0.6445 (0.6445)  loss_bbox_0_unscaled: 0.0058 (0.0058)  loss_giou_0_unscaled: 0.3036 (0.3036)  cardinality_error_0_unscaled: 265.5000 (265.5000)  loss_ce_1_unscaled: 0.6094 (0.6094)  loss_bbox_1_unscaled: 0.0056 (0.0056)  loss_g

[2026-04-14 21:37:45] [INFO] rf-detr - Epoch: [3]  [   0/1439]  eta: 1:17:19  lr: 0.000100  class_error: -0.00  loss: 5.0296 (5.0296)  loss_ce: 0.4873 (0.4873)  loss_bbox: 0.0283 (0.0283)  loss_giou: 0.4502 (0.4502)  loss_ce_0: 0.5251 (0.5251)  loss_bbox_0: 0.0310 (0.0310)  loss_giou_0: 0.4839 (0.4839)  loss_ce_1: 0.4911 (0.4911)  loss_bbox_1: 0.0295 (0.0295)  loss_giou_1: 0.4706 (0.4706)  loss_ce_2: 0.4908 (0.4908)  loss_bbox_2: 0.0283 (0.0283)  loss_giou_2: 0.4348 (0.4348)  loss_ce_enc: 0.5174 (0.5174)  loss_bbox_enc: 0.0346 (0.0346)  loss_giou_enc: 0.5268 (0.5268)  loss_ce_unscaled: 0.4873 (0.4873)  class_error_unscaled: -0.0000 (-0.0000)  loss_bbox_unscaled: 0.0057 (0.0057)  loss_giou_unscaled: 0.2251 (0.2251)  cardinality_error_unscaled: 3871.5000 (3871.5000)  loss_ce_0_unscaled: 0.5251 (0.5251)  loss_bbox_0_unscaled: 0.0062 (0.0062)  loss_giou_0_unscaled: 0.2420 (0.2420)  cardinality_error_0_unscaled: 3869.5000 (3869.5000)  loss_ce_1_unscaled: 0.4911 (0.4911)  loss_bbox_1_unscale

[2026-04-14 23:10:16] [INFO] rf-detr - Test:  [   0/2880]  eta: 0:19:51  class_error: 0.00  loss: 6.1614 (6.1614)  loss_ce: 0.6172 (0.6172)  loss_bbox: 0.0266 (0.0266)  loss_giou: 0.5358 (0.5358)  loss_ce_0: 0.6523 (0.6523)  loss_bbox_0: 0.0290 (0.0290)  loss_giou_0: 0.5792 (0.5792)  loss_ce_1: 0.6094 (0.6094)  loss_bbox_1: 0.0282 (0.0282)  loss_giou_1: 0.5757 (0.5757)  loss_ce_2: 0.6172 (0.6172)  loss_bbox_2: 0.0274 (0.0274)  loss_giou_2: 0.5515 (0.5515)  loss_ce_enc: 0.6250 (0.6250)  loss_bbox_enc: 0.0336 (0.0336)  loss_giou_enc: 0.6534 (0.6534)  loss_ce_unscaled: 0.6172 (0.6172)  class_error_unscaled: 0.0000 (0.0000)  loss_bbox_unscaled: 0.0053 (0.0053)  loss_giou_unscaled: 0.2679 (0.2679)  cardinality_error_unscaled: 264.5000 (264.5000)  loss_ce_0_unscaled: 0.6523 (0.6523)  loss_bbox_0_unscaled: 0.0058 (0.0058)  loss_giou_0_unscaled: 0.2896 (0.2896)  cardinality_error_0_unscaled: 265.5000 (265.5000)  loss_ce_1_unscaled: 0.6094 (0.6094)  loss_bbox_1_unscaled: 0.0056 (0.0056)  loss_g

[2026-04-14 23:27:28] [INFO] rf-detr - Epoch: [4]  [   0/1439]  eta: 1:15:28  lr: 0.000100  class_error: 0.00  loss: 6.4124 (6.4124)  loss_ce: 0.5869 (0.5869)  loss_bbox: 0.0371 (0.0371)  loss_giou: 0.6301 (0.6301)  loss_ce_0: 0.6124 (0.6124)  loss_bbox_0: 0.0396 (0.0396)  loss_giou_0: 0.6319 (0.6319)  loss_ce_1: 0.5941 (0.5941)  loss_bbox_1: 0.0377 (0.0377)  loss_giou_1: 0.6346 (0.6346)  loss_ce_2: 0.5905 (0.5905)  loss_bbox_2: 0.0374 (0.0374)  loss_giou_2: 0.6426 (0.6426)  loss_ce_enc: 0.5993 (0.5993)  loss_bbox_enc: 0.0437 (0.0437)  loss_giou_enc: 0.6945 (0.6945)  loss_ce_unscaled: 0.5869 (0.5869)  class_error_unscaled: 0.0000 (0.0000)  loss_bbox_unscaled: 0.0074 (0.0074)  loss_giou_unscaled: 0.3150 (0.3150)  cardinality_error_unscaled: 3878.0000 (3878.0000)  loss_ce_0_unscaled: 0.6124 (0.6124)  loss_bbox_0_unscaled: 0.0079 (0.0079)  loss_giou_0_unscaled: 0.3159 (0.3159)  cardinality_error_0_unscaled: 3871.0000 (3871.0000)  loss_ce_1_unscaled: 0.5941 (0.5941)  loss_bbox_1_unscaled: 

<font color='blue'>debug/investigation cells we used to figure out how RF-DETR's callback system works.

In [ ]:
# Check engine.py for callback/hook support
import inspect
from rfdetr import engine
print(inspect.getsource(engine))

# ------------------------------------------------------------------------
# RF-DETR
# Copyright (c) 2025 Roboflow. All Rights Reserved.
# Licensed under the Apache License, Version 2.0 [see LICENSE for details]
# ------------------------------------------------------------------------
# Copied and modified from LW-DETR (https://github.com/Atten4Vis/LW-DETR)
# Copyright (c) 2024 Baidu. All Rights Reserved.
# ------------------------------------------------------------------------
# Conditional DETR
# Copyright (c) 2021 Microsoft. All Rights Reserved.
# Licensed under the Apache License, Version 2.0 [see LICENSE for details]
# ------------------------------------------------------------------------
# Copied from DETR (https://github.com/facebookresearch/detr)
# Copyright (c) Facebook, Inc. and its affiliates. All Rights Reserved.
# ------------------------------------------------------------------------

"""
Train and eval functions used in main.py
"""

import math
import random
from ty

In [ ]:
# Check detr.py train method signature
from rfdetr import RFDETRLarge
print(inspect.getsource(RFDETRLarge.train))

    def train(self, **kwargs):
        """
        Train an RF-DETR model.
        """
        config = self.get_train_config(**kwargs)
        self.train_from_config(config, **kwargs)



In [ ]:
#check train_from_config which is where callbacks are actually passed:
#show us exactly how to pass callbacks into training.
#self.callbacks is a dict on the model object, and you append to self.callbacks["on_fit_epoch_end"].
import inspect
from rfdetr import RFDETRLarge
print(inspect.getsource(RFDETRLarge.train_from_config))

    def train_from_config(self, config: TrainConfig, **kwargs):
        if config.dataset_file == "roboflow":
            class_names = self._load_classes(config.dataset_dir)
            num_classes = len(class_names) + 1
            self.model.class_names = class_names
        elif config.dataset_file == "yolo":
            class_names = self._load_classes(config.dataset_dir)
            num_classes = len(class_names)
            self.model.class_names = class_names
        elif config.dataset_file == "coco":
            class_names = COCO_CLASSES
            num_classes = 90
        else:
            raise ValueError(f"Invalid dataset file: {config.dataset_file}")

        if self.model_config.num_classes != num_classes:
            logger.warning(f"Reinitializing your detection head with {num_classes} classes.")
            self.model.reinitialize_detection_head(num_classes)

        train_config = config.model_dump()
        model_config = self.model_config.model_dump()
        mod

In [ ]:
#  Diagnostic:Check if training process is still alive
# GPU-Util >0% + memory used = still running
# GPU-Util 0% + 0MiB = crashed
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

Difference between:

!gsutil ls gs://mywaymo-perdataset-2026/checkpoints/ | grep rfdetr | sort

and

!gsutil ls gs://mywaymo-perdataset-2026/checkpoints/ | grep rfdetr | sort -t_ -k3 -n

|     | sort    | sort -t_ -k3 -n |
|-----|---------|-----------------|
|Method| Alphabetical| Numerical |
|epoch_9 vs epoch_10|9 comes AFTER 10 ❌|9 comes BEFORE 10 ✅|
|Resume from correct epoch|❌ Risk of wrong epoch|✅ Always correct

In [9]:
# Check what checkpoints we have
!gsutil ls gs://mywaymo-perdataset-2026/checkpoints/ | grep rfdetr | sort -t_ -k3 -n

gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_0.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_1.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_2.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_3.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_4.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_5.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_6.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_7.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_8.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_9.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_10.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_11.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_12.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_13.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_14.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_15.pth
gs://mywaymo-perdataset-2026/check

In [ ]:
#shows the last 5 checkpoints — tells us how many epochs completed before crash
!gsutil ls gs://mywaymo-perdataset-2026/checkpoints/ | grep rfdetr | sort -t_ -k3 -n | tail -3

gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_44.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_45.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_46.pth


In [12]:
#check what's in the models folder:
!gsutil ls gs://mywaymo-perdataset-2026/models/ | grep rfdetr

gs://mywaymo-perdataset-2026/models/rfdetr_150_training_history.csv


In [13]:
#Check what was saved before it died:
!gsutil ls gs://mywaymo-perdataset-2026/checkpoints/ | grep rfdetr | sort
!gsutil ls gs://mywaymo-perdataset-2026/models/ | grep csv

gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_0.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_10.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_11.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_12.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_13.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_14.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_15.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_16.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_17.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_18.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_19.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_1.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_20.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_21.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_22.pth
gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_23.pth
gs://mywaymo-perdataset-20

In [14]:
#save the best checkpoint as the official final model.
import subprocess

# Copy epoch 52 (last = likely best) as final model
subprocess.run(["gsutil", "cp",
    "gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_52.pth",
    "gs://mywaymo-perdataset-2026/models/waymo_rfdetr_150_last.pth"
])

# Also find best by checking checkpoint_best files in output dir
!gsutil ls /content/runs/rfdetr_150/ 2>/dev/null || echo "Local output dir gone"

print("✅ Last model saved to GCS")

Local output dir gone
✅ Last model saved to GCS


In [15]:
#Find the highest mAP@0.50 value and note which epoch it was.
#If the output is too long to scroll, check the CSV:
!gsutil cp gs://mywaymo-perdataset-2026/models/rfdetr_150_training_history.csv /content/
import pandas as pd
df = pd.read_csv('/content/rfdetr_150_training_history.csv')
print(f"Total rows: {len(df)}")
print(df.to_string())

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://mywaymo-perdataset-2026/models/rfdetr_150_training_history.csv...
/ [1 files][ 22.0 KiB/ 22.0 KiB]                                                
Operation completed over 1 objects/22.0 KiB.                                     
Total rows: 4
   train_lr  train_class_error  train_loss  train_loss_ce  train_loss_bbox  train_loss_giou  train_loss_ce_0  train_loss_bbox_0  train_loss_giou_0  train_loss_ce_1  train_loss_bbox_1  train_loss_giou_1  train_loss_ce_2  train_loss_bbox_2  train_loss_giou_2  train_loss_ce_enc  train_loss_bbox_enc  train_loss_giou_enc  train_loss_ce_unscaled  train_class_error_unscaled  train_loss_bbox_unscaled  train_loss_giou_unscaled  train_cardinality_error_unscaled  train_loss_ce_0_unscaled  tr

In [16]:
#save the best model:
# epoch 3 in CSV = last resume run's epoch 3
# Best checkpoint is rfdetr_epoch_49.pth (epoch 46 + 3)
!gsutil cp gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_49.pth \
           gs://mywaymo-perdataset-2026/models/waymo_rfdetr_150_best.pth

print("✅ Best model saved!")

# Verify both models exist
!gsutil ls gs://mywaymo-perdataset-2026/models/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://mywaymo-perdataset-2026/checkpoints/rfdetr_epoch_49.pth [Content-Type=application/octet-stream]...
/ [1 files][513.9 MiB/513.9 MiB]                                                
Operation completed over 1 objects/513.9 MiB.                                    
✅ Best model saved!
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://mywaymo-perdataset-2026/models/BoxF1_curve.png
gs://mywaymo-perdataset-2026/models/BoxPR_curve.png
gs://mywaymo-perdataset-2026/models/BoxP_curve.png
gs://mywaymo-perdataset-2026/mod